Imports


In [10]:
from collections import deque
import requests
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
import time
import pickle
from pathlib import Path
from sentence_transformers import SentenceTransformer
import numpy as np





/opt/anaconda3/envs/yolo1-env1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


crawler


In [2]:
def crawl_products(start_points):
    visited = set()
    seen = set(start_points.keys())
    
    queue = deque([(url, 0, max_depth) for url, max_depth in start_points.items()])
    
    product_handles = set()
    base_domain = urlparse(next(iter(start_points))).netloc

    while queue:
        url, depth, max_depth = queue.popleft()
        
        print(f"[DEPTH {depth}/{max_depth}] A visitar: {url}")

        if depth > max_depth:
            continue
        
        visited.add(url)

        try:
            response = requests.get(url, timeout=5)
            soup = BeautifulSoup(response.text, 'html.parser')

            for link in soup.find_all('a', href=True):
                full_link = urljoin(url, link['href'])
                parsed = urlparse(full_link)

                
                if parsed.netloc != base_domain:
                    continue

                
                if any(x in full_link for x in [
                    "cart", "login", "wishlist", "translate",
                    "account", "checkout", "search"
                ]):
                    continue

                path = parsed.path

               
                if path.startswith("/product/"):
                    handle = path.split("/product/")[1]
                    handle = handle.split("?")[0]  # limpar params
                    handle = handle.strip("/")

                    if handle not in product_handles:
                        product_handles.add(handle)
                        print(f"  Produto encontrado: {handle}")

                    continue 

                
                if full_link not in seen:
                    seen.add(full_link)
                    queue.append((full_link, depth + 1, max_depth))

            print(f"  -> Total produtos: {len(product_handles)}")
            print(f"  -> Links vistos: {len(seen)}")

            time.sleep(1)

        except Exception as e:
            print(f"[ERRO] {url}: {e}")

    return product_handles


variaveis

In [3]:
start_points = {
        "https://www.inocos.com/": 1,
        "https://www.inocos.com/category": 2,
    }

products = crawl_products(start_points)


[DEPTH 0/1] A visitar: https://www.inocos.com/
  Produto encontrado: mystery-box-inocos
  Produto encontrado: box-verniz-gel-cateye-celestial-inocos
  Produto encontrado: iman-magnetico-inocos-cateye-5-em-1
  Produto encontrado: verniz-gel-inocos-cateye-aurora-boreal
  Produto encontrado: verniz-gel-inocos-cateye-cometa
  Produto encontrado: top-coat-inocos-cafe-natural
  Produto encontrado: multi-portable-nail-drill-inocos
  Produto encontrado: almofada-inocos-manicure-apoio-de-bracos
  Produto encontrado: box-glitter-biodegradavel-bioglitter-inocos
  Produto encontrado: glow-pro-nail-desk-lamp-candeeiro-inocos
  Produto encontrado: kit-complementos-manicure-russa-combinada-inocos
  -> Total produtos: 11
  -> Links vistos: 153
[DEPTH 0/2] A visitar: https://www.inocos.com/category
  -> Total produtos: 11
  -> Links vistos: 153
[DEPTH 1/1] A visitar: https://www.inocos.com/page/pontos-de-venda
  -> Total produtos: 11
  -> Links vistos: 184
[DEPTH 1/1] A visitar: https://www.inocos.com/

In [4]:
print("\n Produtos encontrados:")
for p in products:
    print(p)

print(f"\nTotal: {len(products)} produtos")


 Produtos encontrados:
like-gel-inocos-230-castanho-torrado
like-gel-inocos-214-rosa-suave-leitoso
builder-gel-inocos-rosa-leitoso-intenso-de-media-viscosidade-50g
neutral-inocos-inibidor-de-odor-aroma-groselha-ativa
verniz-gel-inocos-aqua-dream
like-gel-inocos-193-esmeralda-pastel
kit-inocos-solid-gel
flocos-de-luz-inocos-branco
nutri-base-inocos
verniz-gel-inocos-sapatos-de-cristal
saco-de-papel-inocos-tamanho-m
easy-perfect-inocos-vintage-rose-30g
like-gel-inocos-222-rosa-intenso-cintilante
verniz-gel-inocos-limonada
like-gel-inocos-182-castanho-barro
spray-inocos-pos-depilatorio
nails-decor-inocos-no7-pearls
oleo-de-cuticulas-inocos-de-morango-15ml
nails-decor-inocos-no2-lovers
verniz-gel-inocos-criar-raizes
cover-up-inocos-avela-50g
verniz-gel-inocos-maria-viana
oleo-de-cuticulas-inocos-de-framboesa-15ml
copo-de-vidro-inocos
verniz-gel-inocos-acredita
verniz-gel-inocos-dinossauro
glitter-xi-coracao-inocos-branco-holografico
cleaner-inocos-150ml
verniz-gel-inocos-carrossel-magico


Pré-processamento dos dados


In [5]:
def clean_handle(handle):
    remove_words = {"inocos", "hifans"}
    
    parts = handle.lower().split("-")
    cleaned = [p for p in parts if p not in remove_words]
    
    return " ".join(cleaned)

variaveis


In [6]:
dados_processados = [clean_handle(p) for p in products]
print(dados_processados)

['like gel 230 castanho torrado', 'like gel 214 rosa suave leitoso', 'builder gel rosa leitoso intenso de media viscosidade 50g', 'neutral inibidor de odor aroma groselha ativa', 'verniz gel aqua dream', 'like gel 193 esmeralda pastel', 'kit solid gel', 'flocos de luz branco', 'nutri base', 'verniz gel sapatos de cristal', 'saco de papel tamanho m', 'easy perfect vintage rose 30g', 'like gel 222 rosa intenso cintilante', 'verniz gel limonada', 'like gel 182 castanho barro', 'spray pos depilatorio', 'nails decor no7 pearls', 'oleo de cuticulas de morango 15ml', 'nails decor no2 lovers', 'verniz gel criar raizes', 'cover up avela 50g', 'verniz gel maria viana', 'oleo de cuticulas de framboesa 15ml', 'copo de vidro', 'verniz gel acredita', 'verniz gel dinossauro', 'glitter xi coracao branco holografico', 'cleaner 150ml', 'verniz gel carrossel magico', 'kit nail art', 'verniz gel preto 90', 'placa de carimbos especial natal 1', 'solid tricolor gel 02 white fantasy', 'glitter xi coracao ros

## Criar embeddings e guardar tabela dos dados originais


In [15]:
DATA_DIR = Path("/Users/tomassilveira/Desktop/projeto/whatapp_to_excel/data")
DATA_DIR.mkdir(exist_ok=True)

model = SentenceTransformer("all-MiniLM-L6-v2")

with open(DATA_DIR / "prod.pkl", "rb") as f:
    produtos = pickle.load(f)

with open(DATA_DIR / "prod.pkl", "wb") as f:
    pickle.dump(dados_processados, f)

produtos_norm = [p.lower() for p in produtos]

embeddings = model.encode(produtos_norm, convert_to_numpy=True)
np.save(DATA_DIR / "emb_prod.npy", embeddings)

print(" Embeddings criados!")
print("Produtos guardados!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7006.27it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Embeddings criados!
Produtos guardados!


## guardar_tabela

In [24]:
DATA_DIR = Path.cwd().parent/"data"
DATA_DIR.mkdir(exist_ok=True)

with open(DATA_DIR / "prod.pkl", "wb") as f:
    pickle.dump(dados_processados, f)

print("Produtos guardados!")

Produtos guardados!
